# Práctica 5: Redes neuronales y backpropagation desde cero

**Fundamentos matemáticos del aprendizaje profundo** · Departamento de Matemática, FCEN-UBA

Material de referencia: capítulos 6 de las notas del curso. Convenciones: $y=\varphi(w\cdot x+b)$, $\sigma$ es la logística, $\eta$ la tasa de aprendizaje, $w_{ij}$ conecta la entrada $i$ con la salida $j$.

Cada práctica tiene una **Parte A** (ejercicios teóricos, para resolver en papel) y una **Parte B** (computacional). Las celdas marcadas `# TODO` son para completar.

## Parte A. Ejercicios teóricos

**A1.** Sea una red con capas $a^{(0)}=x$, $z^{(l)}=W^{(l)}a^{(l-1)}+b^{(l)}$, $a^{(l)}=\varphi(z^{(l)})$ y costo $C$. Definiendo $\delta^{(l)}=\partial C/\partial z^{(l)}$, mostrar
$$\delta^{(L)}=\nabla_{a^{(L)}}C\odot\varphi'(z^{(L)}),\quad \delta^{(l)}=\big((W^{(l+1)})^\top\delta^{(l+1)}\big)\odot\varphi'(z^{(l)}),$$
$$\partial C/\partial W^{(l)}=\delta^{(l)}(a^{(l-1)})^\top,\quad \partial C/\partial b^{(l)}=\delta^{(l)}.$$

**A2.** Contar el número de operaciones (multiplicaciones) del cálculo del gradiente por backpropagation y compararlo con diferencias finitas. ¿Por qué se llama "modo reverso"?

**A3.** Hallar una red con una capa oculta de perceptrones que aprenda el mapa de la práctica (dado en clase) y dar pesos y sesgos de todos los perceptrones. Dibujar el esquema.

**A4.** (Simetría) Si todos los pesos de la capa oculta se inicializan con el mismo valor, mostrar que las neuronas ocultas permanecen idénticas durante todo el entrenamiento por GD. ¿Qué implica para la inicialización?

**A5.** Para la red $y=\sum_{j=1}^m c_j\,\sigma(w_jx+b_j)$ con costo cuadrático sobre un dato, escribir explícitamente $\partial C/\partial c_j$, $\partial C/\partial w_j$, $\partial C/\partial b_j$.

## Parte B. Computacional

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize']=(6,4)
rng=np.random.default_rng(0)

### B1. Red completamente conectada con backprop
Completar la clase. Usar activación `tanh`, costo cuadrático y mini-batches. Los datos son matrices con una muestra por fila, por lo que se usan pesos $W^{(l)}\in\mathbb R^{n_{l-1}\times n_l}$ y $Z=AW+b$ (equivalente a lo de A1 trasponiendo).

In [ ]:
class Red:
    def __init__(self,capas,rng):
        self.W=[rng.standard_normal((a,b))*np.sqrt(1/a) for a,b in zip(capas[:-1],capas[1:])]
        self.b=[np.zeros(b) for b in capas[1:]]
    def forward(self,X):
        A=[X]; Zs=[]
        for l,(W,b) in enumerate(zip(self.W,self.b)):
            Z=A[-1]@W+b; Zs.append(Z)
            A.append(Z if l==len(self.W)-1 else np.tanh(Z))   # ultima capa lineal
        return A,Zs
    def grad(self,X,Y):
        A,Zs=self.forward(X); N=len(X)
        delta=(A[-1]-Y)/N                       # dC/dZ para C=1/(2N) sum |a-y|^2
        gW=[None]*len(self.W); gb=[None]*len(self.W)
        for l in reversed(range(len(self.W))):
            gW[l]=A[l].T@delta; gb[l]=delta.sum(0)
            if l>0: delta=(delta@self.W[l].T)*(1-A[l]**2)   # tanh'=1-tanh^2
        return gW,gb
    def costo(self,X,Y): return 0.5*np.mean(np.sum((self.forward(X)[0][-1]-Y)**2,axis=1))

### B2. Verificación del gradiente
Comparar `grad` con diferencias finitas centradas sobre todos los parámetros. El error relativo debe ser $\lesssim10^{-7}$.

In [ ]:
red=Red([2,5,4,1],rng)
X=rng.standard_normal((10,2)); Y=rng.standard_normal((10,1))
gW,gb=red.grad(X,Y)
h=1e-6; err=0
for W,g in zip(red.W,gW):
    for idx in np.ndindex(W.shape):
        old=W[idx]; W[idx]=old+h; cp=red.costo(X,Y); W[idx]=old-h; cm=red.costo(X,Y); W[idx]=old
        err=max(err,abs((cp-cm)/(2*h)-g[idx]))
print('max error absoluto (pesos):',err)
# TODO: lo mismo para los sesgos

### B3. Ajustar una función 1D
Entrenar `Red([1,20,20,1])` para aproximar $f(x)=\sin(3x)$ en $[-1,1]$ con GD (mini-batch). Graficar la aproximación y la curva de costo.

In [ ]:
# TODO

### B4. XOR y simetría
(a) Entrenar `Red([2,2,1])` en XOR con tanh y pesos aleatorios. (b) Repetir inicializando **todos** los pesos de cada capa con el mismo valor y verificar A4: las dos neuronas ocultas coinciden siempre. ¿Qué se obtiene?

In [ ]:
# TODO

### B5. Costo computacional
Medir el tiempo de `grad` vs. diferencias finitas al crecer el número de parámetros. Graficar en escala log-log.

In [ ]:
# TODO